In [1]:
import cuml

from cuml.ensemble import RandomForestRegressor

from sklearn import ensemble
from sklearn.ensemble import ExtraTreesRegressor

import sklearn

import matplotlib.pyplot as plt

import xgboost as xgb
from lightgbm import LGBMRegressor

import torch

import numpy as np
import cupy as cp

import pandas as pd
from pathlib import Path
import gc

from sklearn.metrics import r2_score

import time

In [2]:
randomf = True
extrat = True
xgboost = True
lgboostm = True

In [3]:
modelsLib = {
    "rf": 1,
    "rfGPU": 1,
    "et": 1,
    "etGPU": 1,
    "xgb": 1,
    "xgbGPU": 1,
    "lgbm": 1,
    "lgbmGPU": 1
}

In [4]:
ET = {
    "model": ExtraTreesRegressor,
    "n_estimators": [50, 50, 50],
    "max_depth": [21, 21, 1],
    "min_samples_leaf": [4, 4, 2],
    "max_features": [1.0, 1.0, 0.1],
    "results": []
}

X = {
    "model": xgb,
    "n_estimators": [500, 500, 50],
    "max_depth": [15, 15, 1],
    "min_child_weight": [3, 3, 1],
    "results": []
}

LGBM = {
    "model": LGBMRegressor,
    "n_estimators": [450, 450, 50],
    "min_child_samples": [15, 15, 10],
    "num_leaves": [256, 256, 2],
    "colsample_bytree": [0.7, 0.7, 0.1],
    "results": [],
    "trials": []
}

In [5]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/trees-gt-aoi.csv")

#making sure
datasetPath.is_file()

True

In [6]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.210754,0.000179,0.000229,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.2
1,0.123970,0.000280,0.000321,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.2
2,0.092202,0.000341,0.000383,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.2
3,0.191730,0.000242,0.000306,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.2
4,0.145418,0.000262,0.000312,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.2


In [7]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.2 , 0.2 , 0.2 , ..., 0.26, 0.26, 0.26],
      shape=(333760,), dtype=float32)

In [8]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.drop("sunRoll", axis=1)
inputs = inputs.drop("camRoll", axis=1)

inputs.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunPitch,sunYaw,camPitch,camYaw
0,0.210754,0.000179,0.000229,1136.471,1031.228,0.0,-71.945084,19.489212,-19.258,-0.0
1,0.123970,0.000280,0.000321,1136.471,1031.228,1.0,-71.945084,19.489212,-19.258,-0.0
2,0.092202,0.000341,0.000383,1136.471,1031.228,2.0,-71.945084,19.489212,-19.258,-0.0
3,0.191730,0.000242,0.000306,1136.471,1031.228,3.0,-71.945084,19.489212,-19.258,-0.0
4,0.145418,0.000262,0.000312,1136.471,1031.228,4.0,-71.945084,19.489212,-19.258,-0.0


In [9]:
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.1075405e-01,  1.7945077e-04,  2.2896443e-04, ...,
         1.9489212e+01, -1.9257999e+01, -0.0000000e+00],
       [ 1.2397036e-01,  2.7967661e-04,  3.2067925e-04, ...,
         1.9489212e+01, -1.9257999e+01, -0.0000000e+00],
       [ 9.2202291e-02,  3.4122023e-04,  3.8319931e-04, ...,
         1.9489212e+01, -1.9257999e+01, -0.0000000e+00],
       ...,
       [ 2.3046868e-01,  1.0813908e-03,  1.6539922e-03, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02],
       [ 2.3434559e-01,  9.0364768e-04,  1.3691875e-03, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02],
       [ 2.4411118e-01,  8.7809877e-04,  1.3535006e-03, ...,
         1.0435840e+02, -2.2218000e+01, -1.8000000e+02]],
      shape=(333760, 10), dtype=float32)

In [10]:
n_total = len(inputs)

n_train = int(0.9*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]

In [11]:
mask_high = val_inputs[:, 4] > 100
mask_low = ~mask_high

val_inputs_high = val_inputs[mask_high]
val_labels_high = val_labels[mask_high]

val_inputs_low = val_inputs[mask_low]
val_labels_low = val_labels[mask_low]

In [12]:
ne = ET["n_estimators"][0]
md = ET["max_depth"][0]
nb = ET["min_samples_leaf"][0]
mf = ET["max_features"][0]

modelET = ET["model"](n_estimators=ne, max_depth=md, min_samples_leaf=nb, max_features=mf)
modelET.fit(train_inputs, train_labels)

y_pred = modelET.predict(val_inputs)
y_pred_shadows = modelET.predict(val_inputs_low)
y_pred_clear = modelET.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))


overall: RelRMSE: 5.5694112951867325%, r2 0.9276709632621387
shadows: RelRMSE: 4.918658021867711%, r2 0.9375277574668405
clear: RelRMSE: 5.739377427146676%, r2 0.925148176268467


In [13]:
ne = X["n_estimators"][0]
md = X["max_depth"][0]
mc = X["min_child_weight"][0]

modelXGB = X["model"].XGBRegressor(n_estimators=ne, max_depth=md, min_child_weight=mc, learning_rate=0.05, tree_method="gpu_hist", gpu_id=0)
modelXGB.fit(train_inputs, train_labels)

y_pred = modelXGB.predict(val_inputs)
y_pred_shadows = modelXGB.predict(val_inputs_low)
y_pred_clear = modelXGB.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))

overall: RelRMSE: 6.354290551844054%, r2 0.9058482646942139
shadows: RelRMSE: 6.206794724854781%, r2 0.9005216360092163
clear: RelRMSE: 6.395215543530656%, r2 0.9070641994476318


In [14]:
ne = LGBM["n_estimators"][0]
mcs = LGBM["min_child_samples"][0]
nl = LGBM["num_leaves"][0]
cb = LGBM["colsample_bytree"][0]

modelLGBM = LGBM["model"](n_estimators=ne, min_child_samples=mcs, num_leaves=nl-1, colsample_bytree=cb)
modelLGBM.fit(train_inputs, train_labels)

y_pred = modelLGBM.predict(val_inputs)
y_pred_shadows = modelLGBM.predict(val_inputs_low)
y_pred_clear = modelLGBM.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007401 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1636
[LightGBM] [Info] Number of data points in the train set: 300384, number of used features: 10
[LightGBM] [Info] Start training from score 0.305386
overall: RelRMSE: 6.783651155160313%, r2 0.8926947151422432
shadows: RelRMSE: 6.754368660929666%, r2 0.8821950982992326
clear: RelRMSE: 6.79187496601623%, r2 0.8951780631758615
